In [1]:
# imports
import numpy as np
import torch
from er_evaluation import pairwise_f, pairwise_precision, pairwise_recall
from torch.utils.data import DataLoader, ConcatDataset, random_split, TensorDataset
from dataset_utils.Cresci17 import Cresci17, Cresci17SetTypes
from transformers import DistilBertTokenizer, AutoModel
import math
import re
import hdbscan
import matplotlib.pyplot as plt
import umap
from sklearn.metrics import confusion_matrix, adjusted_rand_score, normalized_mutual_info_score, accuracy_score, \
    f1_score, recall_score
from sklearn.preprocessing import RobustScaler
from hdbscan import approximate_predict
import joblib
from torch import nn
from pipeline_utils import create_profile_vector, create_tweet_vectors, train_classifier, test_classifier
from ClassificationHeads import BaselineClassificationHead

c:\Users\Hamouda\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load dataset
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# use generator to get consistent dataset split between re-runs
generator = torch.Generator().manual_seed(42)
social_spam1_train, social_spam1_test = random_split(Cresci17(root="./datasets", subset_type=Cresci17SetTypes.SOCIAL_SPAM_1),[0.8,0.2], generator=generator)
fake_follower_train, fake_follower_test = random_split(Cresci17(root="./datasets", subset_type=Cresci17SetTypes.FAKE_FOLLOWER),[0.8,0.2], generator=generator)
genuine_user_train, genuine_user_test = random_split(Cresci17(root="./datasets", subset_type=Cresci17SetTypes.GENUINE_USER),[0.8,0.2], generator=generator)
traditional_spam = Cresci17(root="./datasets", subset_type=Cresci17SetTypes.TRADITIONAL_SPAM_1)

train_dataloader = DataLoader(ConcatDataset([social_spam1_train,genuine_user_train,fake_follower_train]), batch_size=None, batch_sampler=None, shuffle=True, pin_memory=True)
test_dataloader = DataLoader(ConcatDataset([social_spam1_test,genuine_user_test,fake_follower_test]), batch_size=None, batch_sampler=None, shuffle=True, pin_memory=True)

d:\Ulm\NLP AND NLM\Project\-Social-Media-Bot-Detection-with-Continuous-Learning\dataset_utils\Cresci17.py:80: DtypeWarning: Columns (0: favorite_count) have mixed types. Specify dtype option on import or set low_memory=False.
  self.tweet_data = pd.read_csv(tweet_data_path, encoding="latin-1", dtype={'id': str},


In [3]:


# initiate models
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
dim_reducer = umap.UMAP(n_components=15, n_neighbors=20, min_dist=0.1, metric="cosine")
clusterer = hdbscan.HDBSCAN(min_cluster_size=60, min_samples=20, prediction_data=True, cluster_selection_epsilon=0.5)
scaler = RobustScaler(with_centering=False)
classifier = BaselineClassificationHead(23,10).to(device)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6058.68it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
from label_remapper import LabelRemapper
from continual_experiment_manager import ContinualExperimentManager
from ClassificationHeads import ProgressiveNeuralNetworkClassifier

In [5]:
DATASET_ROOT = "./datasets"   
MAX_TWEETS_PER_USER = 20
TWEET_BATCH_SIZE = 32
UMAP_COMPONENTS = 15
EPOCHS = 15
LEARNING_RATE = 0.01
USE_INTERVENTION_OVERRIDE = True
GENERATOR = torch.Generator().manual_seed(42)

In [6]:
TASK_DEFINITIONS = [
    (Cresci17SetTypes.GENUINE_USER,       "GenuineUser"),
    (Cresci17SetTypes.FAKE_FOLLOWER,      "FakeFollower"),
    (Cresci17SetTypes.SOCIAL_SPAM_1,      "SocialSpam1"),
    (Cresci17SetTypes.TRADITIONAL_SPAM_1, "TradSpam1"),
]

task_splits = []
for subset_type, task_name in TASK_DEFINITIONS:
    dataset = Cresci17(root=DATASET_ROOT, subset_type=subset_type)
    train_split, test_split = random_split(dataset, [0.8, 0.2], generator=GENERATOR)
    native_label = subset_type.value
    task_splits.append((train_split, test_split, task_name, native_label))
    print(f"Loaded '{task_name}': {len(train_split)} train, {len(test_split)} test (native label={native_label})")

Loaded 'GenuineUser': 2779 train, 694 test (native label=1)


d:\Ulm\NLP AND NLM\Project\-Social-Media-Bot-Detection-with-Continuous-Learning\dataset_utils\Cresci17.py:80: DtypeWarning: Columns (0: favorite_count) have mixed types. Specify dtype option on import or set low_memory=False.
  self.tweet_data = pd.read_csv(tweet_data_path, encoding="latin-1", dtype={'id': str},


Loaded 'FakeFollower': 2681 train, 670 test (native label=2)
Loaded 'SocialSpam1': 793 train, 198 test (native label=3)
Loaded 'TradSpam1': 800 train, 200 test (native label=6)


In [7]:
def embed_split(split, task_name):
    loader = DataLoader(split, batch_size=None, shuffle=False)
    features = []
    labels = []
    print(f"  Embedding {len(split)} samples from '{task_name}'...")
    for i, (sample, label) in enumerate(loader):
        if i % 100 == 0:
            print(f"    {i}/{len(split)}")
        profile_vec = create_profile_vector(sample)
        tweet_vecs = create_tweet_vectors(
            sample, tokenizer=tokenizer, model=model,
            batch_size=TWEET_BATCH_SIZE, max_tweets=MAX_TWEETS_PER_USER, device=device
        )
        tweet_vec = torch.mean(tweet_vecs, dim=0)
        combined = torch.cat([profile_vec, tweet_vec], dim=0)
        features.append(combined.numpy())
        labels.append(int(label))
    return np.array(features), labels

In [8]:
print("Embedding ALL tasks upfront to fit UMAP and Scaler on all classes...")

all_raw_features = []
all_labels = []

for train_split, _, task_name, native_label in task_splits:
    raw, lbls = embed_split(train_split, task_name)
    all_raw_features.append(raw)
    all_labels.extend(lbls)
    print(f"  Done embedding '{task_name}'")

all_raw_features = np.vstack(all_raw_features)
print(f"\nTotal samples for UMAP fitting: {all_raw_features.shape}")

print("Fitting RobustScaler...")
all_scaled = scaler.fit_transform(all_raw_features)

print(f"Fitting UMAP (n_components={UMAP_COMPONENTS})...")
dim_reducer.fit(all_scaled)
print("UMAP fitted on all classes. Done!")

# now split back out task 0 embeddings for the main loop
# (since the main loop reuses umap_features_task0 for task 0)
task0_size = len(task_splits[0][0])  # number of training samples in task 0
task0_scaled = all_scaled[:task0_size]
umap_features_task0 = dim_reducer.transform(task0_scaled)
raw_features_task0 = all_raw_features[:task0_size]
labels_task0 = all_labels[:task0_size]

print(f"Task 0 UMAP shape: {umap_features_task0.shape}")

def transform_features(raw_features):
    scaled = scaler.transform(raw_features)
    return dim_reducer.transform(scaled)

Embedding ALL tasks upfront to fit UMAP and Scaler on all classes...
  Embedding 2779 samples from 'GenuineUser'...
    0/2779
    100/2779
    200/2779
    300/2779
    400/2779
    500/2779
    600/2779
    700/2779
    800/2779
    900/2779
    1000/2779
    1100/2779
    1200/2779
    1300/2779
    1400/2779
    1500/2779
    1600/2779
    1700/2779
    1800/2779
    1900/2779
    2000/2779
    2100/2779
    2200/2779
    2300/2779
    2400/2779
    2500/2779
    2600/2779
    2700/2779
  Done embedding 'GenuineUser'
  Embedding 2681 samples from 'FakeFollower'...
    0/2681
    100/2681
    200/2681
    300/2681
    400/2681
    500/2681
    600/2681
    700/2681
    800/2681
    900/2681
    1000/2681
    1100/2681
    1200/2681
    1300/2681
    1400/2681
    1500/2681
    1600/2681
    1700/2681
    1800/2681
    1900/2681
    2000/2681
    2100/2681
    2200/2681
    2300/2681
    2400/2681
    2500/2681
    2600/2681
  Done embedding 'FakeFollower'
  Embedding 793 samples fro

In [9]:
label_remapper = LabelRemapper()
experiment_manager = ContinualExperimentManager(
    num_tasks=len(task_splits),
    use_intervention_override=USE_INTERVENTION_OVERRIDE # this is task 3 for label mismatch
)

classifier = ProgressiveNeuralNetworkClassifier(
    in_features=UMAP_COMPONENTS,
    output_dim=1,
    dropout_p=0.1
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=LEARNING_RATE)
test_splits_cache = []

In [10]:
for task_idx, (train_split, test_split, task_name, native_label) in enumerate(task_splits):
    print(f"\n{'='*60}")
    print(f"STEP {task_idx}: '{task_name}' (native label={native_label})")
    print(f"{'='*60}")

    # 1. Embed training data
    if task_idx == 0:
        umap_train = umap_features_task0
        native_train_labels = labels_task0
    else:
        raw_train, native_train_labels = embed_split(train_split, task_name)
        umap_train = transform_features(raw_train)

    # 2. Run HDBSCAN
    print(f"Running HDBSCAN on {len(umap_train)} samples...")
    clusterer = hdbscan.HDBSCAN(min_cluster_size=10, prediction_data=True)
    cluster_labels = clusterer.fit_predict(umap_train)

    # 3. Expand classifier if needed
    if task_idx == 0:
        print("Task 0: no expansion needed.")
    else:
        experiment_manager.check_clusterer_and_expand(
            clusterer_labels=cluster_labels,
            ground_truth_labels=native_train_labels,
            classifier=classifier
        )

    # 4. Remap labels
    label_remapper.register(native_label)
    remapped_train_labels = label_remapper.convert(native_train_labels)
    print(f"Label remapper: {label_remapper}")

    # 5. Re-create optimizer after expansion
    optimizer = torch.optim.Adam(classifier.parameters(), lr=LEARNING_RATE)

    # 6. Train
    print(f"Training for {EPOCHS} epochs...")
    train_classifier(classifier, criterion, optimizer, umap_train, remapped_train_labels, epochs=EPOCHS, device=device)

    # 7. Embed and cache test split
    raw_test, native_test_labels = embed_split(test_split, task_name)
    umap_test = transform_features(raw_test)
    remapped_test_labels = label_remapper.convert(native_test_labels)
    test_splits_cache.append((umap_test, remapped_test_labels, task_name))

    # 8. Evaluate on all past test splits
    print(f"Evaluating on all {len(test_splits_cache)} test split(s)...")
    for j, (cached_features, cached_labels, cached_name) in enumerate(test_splits_cache):
        ground_truths, predictions = test_classifier(classifier, cached_features, cached_labels, device=device)
        acc = accuracy_score(ground_truths, predictions)
        f1  = f1_score(ground_truths, predictions, average="macro", zero_division=0)
        print(f"  Task {j} ('{cached_name}'): acc={acc:.4f}, f1={f1:.4f}")
        experiment_manager.record_accuracy(task_index=j, accuracy=acc)

    # 9. Advance
    experiment_manager.advance_to_next_task()

experiment_manager.summary()


STEP 0: 'GenuineUser' (native label=1)
Running HDBSCAN on 2779 samples...
Task 0: no expansion needed.
Label remapper: LabelRemapper({1: 0})
Training for 15 epochs...
Epoch [1/15] - Loss: 0.0000
Epoch [2/15] - Loss: 0.0000
Epoch [3/15] - Loss: 0.0000
Epoch [4/15] - Loss: 0.0000
Epoch [5/15] - Loss: 0.0000
Epoch [6/15] - Loss: 0.0000
Epoch [7/15] - Loss: 0.0000
Epoch [8/15] - Loss: 0.0000
Epoch [9/15] - Loss: 0.0000
Epoch [10/15] - Loss: 0.0000
Epoch [11/15] - Loss: 0.0000
Epoch [12/15] - Loss: 0.0000
Epoch [13/15] - Loss: 0.0000
Epoch [14/15] - Loss: 0.0000
Epoch [15/15] - Loss: 0.0000
  Embedding 694 samples from 'GenuineUser'...
    0/694
    100/694
    200/694
    300/694
    400/694
    500/694
    600/694
Evaluating on all 1 test split(s)...
  Task 0 ('GenuineUser'): acc=1.0000, f1=1.0000

--- Step 0 complete ---
    Avg Accuracy (this step): 1.0000
    Forgetting Measure:       0.0000
    Row scores: ['1.000']

=== Now starting Step 1 ===


STEP 1: 'FakeFollower' (native label=